# Patent Disruption + F/E/G — CD / F,E,G / n_i,n_j,n_k per window

For every US utility patent, computes at four citation windows (**3, 5, 10, full**):
the **Disruption index** (CD), the **F/E/G** shares (Foundation / Extension / Generalization), and the
disruption components **n_i, n_j, n_k**. All of these come from the same per-citer quantities, so they are
computed together in one pass.

## Raw data (directory & structure)
PatentsView **Granted** bulk `.tsv.zip` (downloaded **2026-05-21** from
`https://data.uspto.gov/bulkdata/datasets/pvgpatdis`):

```
/project/jevans/Dawoon/Science of Science/PatentView/Granted/
├── g_patent.tsv.zip              # patent_id, patent_type, patent_date, ...   -> grant year, utility filter
└── g_us_patent_citation.tsv.zip  # patent_id (citing), citation_patent_id (cited), ...
```
Only **utility↔utility** patent citations are kept, and directed `(citing, cited)` pairs are
**de-duplicated** (the bulk file repeats a few) with self-loops dropped — the same two steps
`patent_disruption_app_add.ipynb` applies, so the granted and extended tables are built
identically and agree exactly at grant years neither application citations nor the citation
window can separate.

## Metrics (formulas)
For focal patent $P$, window $W$: let $A_W$ = patents citing $P$ within $W$ years
($0\le y_c-y_P\le W$), $\mathrm{refs}(P)$ = $P$'s references, and for each citer $C$
- $up_C=|\mathrm{refs}(P)\cap\mathrm{refs}(C)|$ (co-cited references, $=e_j$),
- $down_C=|A_W\cap\mathrm{refs}(C)|$ (co-cited descendants, $=e_i$).

**Disruption components** (within $W$): $B_W$ = patents citing a reference of $P$ (window),
$$n_j=\#\{C\in A_W: up_C>0\}=|A_W\cap B_W|,\quad n_i=|A_W|-n_j,\quad n_k=|B_W|-n_j,$$
$$\mathrm{CD}_W=\frac{n_i-n_j}{n_i+n_j+n_k}\in[-1,1].$$

**F/E/G** (per citer, share of $|A_W|$): $up=down=0\Rightarrow$ **G**; $up>down\Rightarrow$ **E**
(Extension); $down>up\Rightarrow$ **F** (Foundation); $up=down>0\Rightarrow$ E,F get 0.5 each.
$F+E+G=1$.

## Output
`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption.parquet` — `patent_id` plus, for each window suffix
(`3`, `5`, `10`, `_all`): `CD{w}, F{w}, E{w}, G{w}, ni{w}, nj{w}, nk{w}`
(e.g. `CD_5, F_5, E_5, G_5, ni_5, nj_5, nk_5`, and `CD_all, F_all, …`). Example preview at the end.

> Runtime: the run is **tqdm-instrumented**, so a live progress bar with **ETA** is shown during the
> main pass (a few hours over ~8.5M patents on this machine).

In [ ]:
# Imports these cells use but never declared -- they were carried in from an
# earlier kernel session, so the notebook only ran interactively.
from tqdm.auto import tqdm
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
DATA_DIR = pv.GRANTED
OUT_DIR  = pv.OUT
OUT_FP   = pv.out('patent_disruption.parquet')
WINDOWS  = [3, 5, 10, -1]
WIN_SFX  = {3: '_3', 5: '_5', 10: '_10', -1: '_all'}
MIN_YEAR, MAX_YEAR = 1976, 2025
pv.preflight('patent_disruption')
print('output ->', OUT_FP)

## 1. Load raw grant years (utility patents)

In [ ]:
%%time
dp = pd.read_csv(os.path.join(DATA_DIR, 'g_patent.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'patent_type', 'patent_date'],
                 dtype={'patent_id': str, 'patent_type': str})
dp['grant_year'] = pd.to_datetime(dp['patent_date'], errors='coerce').dt.year
dp = dp[dp['patent_type'] == 'utility'].dropna(subset=['grant_year'])
dp['grant_year'] = dp['grant_year'].astype(int)
dp = dp[dp['grant_year'].between(MIN_YEAR, MAX_YEAR)]
pid2year = dict(zip(dp['patent_id'], dp['grant_year']))
util = set(pid2year)
print(f'utility patents with grant year: {len(pid2year):,}')
del dp; gc.collect()

## 2. Load raw citations → utility↔utility edges → integer codes

In [ ]:
%%time
cf, ct, seen, t0 = [], [], 0, time.time()
for ch in pd.read_csv(os.path.join(DATA_DIR, 'g_us_patent_citation.tsv.zip'), sep='\t',
                      usecols=['patent_id', 'citation_patent_id'], dtype=str, chunksize=5_000_000):
    ch = ch.dropna(subset=['citation_patent_id'])
    m = ch['patent_id'].isin(util) & ch['citation_patent_id'].isin(util)
    if m.any():
        cf.append(ch['patent_id'].values[m.values]); ct.append(ch['citation_patent_id'].values[m.values])
    seen += len(ch)
citing_pid = np.concatenate(cf); cited_pid = np.concatenate(ct); del cf, ct; gc.collect()
print(f'[{time.time()-t0:.0f}s] scanned {seen:,} rows -> {len(citing_pid):,} utility<->utility edges')

allids = pd.Index(pd.unique(np.concatenate([citing_pid, cited_pid])))
codemap = {p: i for i, p in enumerate(allids)}
c_from = pd.Series(citing_pid).map(codemap).to_numpy(np.int64)
c_to   = pd.Series(cited_pid).map(codemap).to_numpy(np.int64)
del citing_pid, cited_pid, codemap; gc.collect()
uni = np.asarray(allids); n = len(uni)

# De-duplicate directed (citing, cited) pairs, then drop self-loops.
#
# g_us_patent_citation repeats a small number of (citing, cited) rows. Left in, each repeat
# makes the same citer appear twice in refs()/citers(), which inflates n_i / n_j and shifts CD.
# It is only ~0.15% of citer-edges, but it also made this table disagree with
# patent_disruption_app.parquet at grant years where the two networks are provably the same
# graph -- the extended build de-duplicates, this one did not, so the discrepancy looked like a
# bug in the extended pipeline rather than a duplicate-row artifact here. The two steps below
# are exactly the ones patent_disruption_app_add.ipynb applies, so the two tables are now
# built the same way and agree exactly wherever they must.
n_raw = len(c_from)
key = c_from * np.int64(n) + c_to               # n < 2^23, so the packed key fits int64 easily
del c_from, c_to; gc.collect()
key = np.unique(key)                            # de-dup directed (citing, cited) pairs
n_uniq = len(key)
c_from = (key // n).astype(np.int64); c_to = (key % n).astype(np.int64); del key; gc.collect()
loop = c_from != c_to                           # a patent citing itself carries no information
n_self = int((~loop).sum())
c_from = c_from[loop]; c_to = c_to[loop]; del loop; gc.collect()

year = np.array([pid2year[p] for p in uni], dtype=np.int64)
print(f'{n:,} patents in citation graph; {len(c_from):,} unique directed edges '
      f'(from {n_raw:,} raw rows: {n_raw - n_uniq:,} duplicates, {n_self:,} self-loops removed)')

## 3. Unified engine: CD + F/E/G + n_i/n_j/n_k (CSR, all windows in one pass)

In [ ]:
def build_csr(src, dst, n):
    order = np.argsort(src, kind='stable'); s = src[order]; d = dst[order]
    indptr = np.zeros(n + 1, np.int64); np.add.at(indptr, s + 1, 1); np.cumsum(indptr, out=indptr)
    return indptr, d

def isin_sorted(x, s):
    if len(s) == 0 or len(x) == 0:
        return np.zeros(len(x), bool)
    i = np.searchsorted(s, x); i = np.clip(i, 0, len(s) - 1)
    return s[i] == x

def compute(c_from, c_to, year, windows, n):
    out_ptr, out_idx = build_csr(c_from, c_to, n)      # refs(x)
    in_ptr,  in_idx  = build_csr(c_to,  c_from, n)     # citers(x)
    keys = ['cd', 'f', 'e', 'g', 'ni', 'nj', 'nk']
    res = {w: {k: np.full(n, np.nan, np.float32) for k in keys} for w in windows}
    for F in tqdm(range(n), desc='CD+FEG', mininterval=5):     # tqdm shows live ETA
        a0, a1 = in_ptr[F], in_ptr[F + 1]
        if a1 == a0:
            continue                                   # no citers -> undefined
        A = in_idx[a0:a1]; yF = year[F]; yA = year[A]
        refs = out_idx[out_ptr[F]:out_ptr[F + 1]]; Rset = np.sort(refs)
        lens = out_ptr[A + 1] - out_ptr[A]; tot = int(lens.sum())
        seg = np.zeros(len(A), np.int64); np.cumsum(lens[:-1], out=seg[1:])
        pos = np.arange(tot) - np.repeat(seg, lens) + np.repeat(out_ptr[A], lens)
        RC = out_idx[pos]                              # refs of each citer (ragged, concatenated)
        ej = np.add.reduceat(isin_sorted(RC, Rset).astype(np.int64), seg) if tot else np.zeros(len(A), np.int64)
        if len(refs):
            B = np.unique(np.concatenate([in_idx[in_ptr[r]:in_ptr[r + 1]] for r in refs])); B = B[B != F]
        else:
            B = np.empty(0, np.int64)
        yB = year[B] if len(B) else np.empty(0, np.int64)
        for w in windows:
            if w == -1:
                inwin = yA - yF >= 0
                mB = (yB - yF >= 0) if len(B) else np.zeros(0, bool)
            else:
                inwin = (yA - yF >= 0) & (yA - yF <= w)
                mB = ((yB - yF >= 0) & (yB - yF <= w)) if len(B) else np.zeros(0, bool)
            Aw = A[inwin]; N = len(Aw); Bw = int(mB.sum())
            if N == 0 and Bw == 0:
                continue
            Cset = np.sort(Aw)
            ei = np.add.reduceat(isin_sorted(RC, Cset).astype(np.int64), seg) if tot else np.zeros(len(A), np.int64)
            up = ej[inwin]; down = ei[inwin]
            cext = int((up > down).sum()); cfound = int((down > up).sum())
            tie = int(((up == down) & (up > 0)).sum()); cg = int(((up == 0) & (down == 0)).sum())
            nj = int((up > 0).sum()); ni = N - nj; nk = Bw - nj
            denom = ni + nj + nk
            r = res[w]
            r['ni'][F] = ni; r['nj'][F] = nj; r['nk'][F] = nk
            r['cd'][F] = (ni - nj) / denom if denom > 0 else np.nan
            if N > 0:
                r['e'][F] = (cext + 0.5 * tie) / N
                r['f'][F] = (cfound + 0.5 * tie) / N
                r['g'][F] = cg / N
    return res
print('engine ready')

In [ ]:
%%time
res = compute(c_from, c_to, year, WINDOWS, n)
print('done')

## 4. Save output

In [ ]:
cols = {'patent_id': uni.astype(str)}
for w in WINDOWS:
    s = WIN_SFX[w]
    cols[f'CD{s}'] = res[w]['cd']
    cols[f'F{s}']  = res[w]['f'];  cols[f'E{s}'] = res[w]['e'];  cols[f'G{s}'] = res[w]['g']
    cols[f'ni{s}'] = res[w]['ni']; cols[f'nj{s}'] = res[w]['nj']; cols[f'nk{s}'] = res[w]['nk']
out = pd.DataFrame(cols)
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
for w in WINDOWS:
    s = WIN_SFX[w]
    print(f'  CD{s}: defined {out[f"CD{s}"].notna().mean()*100:5.1f}%  mean {out[f"CD{s}"].mean():.4f}  |  '
          f'mean f/e/g = {out[f"F{s}"].mean():.3f}/{out[f"E{s}"].mean():.3f}/{out[f"G{s}"].mean():.3f}')

## 5. Output directory & example result

**File:** `/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption.parquet`
Columns: `patent_id` + `{cd,f,e,g,ni,nj,nk}` × window suffix `{3, 5, 10, _all}`.

In [ ]:
print('columns:', list(out.columns))
print('\nhead:'); display(out.head(10))
print('\ndescribe (cd + f/e/g per window):')
display(out[[c for c in out.columns if c.split('_')[0] in ('CD','F','E','G')]].describe())

## 6. Cohort percentiles of CD — filing year × CPC Section

For every window `h` in 3/5/10/all, `CD_h_pctl` is the **minimum-rank percentile** (0–1) of `CD_h`
among the patents of the same **filing year × CPC Section** (first letter of the primary
`cpc_code`, A–H or Y) that have a non-missing `CD_h`. Ties share the lowest rank of the tie
(`rank()`). `pctl_year` and `pctl_group` record the cohort keys; patents without a filing year or
a Section get missing percentiles. Filing year and Section come from `patent_metadata.parquet`;
filing year is the cohort of the DA analyses (the CD windows themselves run from the grant year).
The cell is self-contained, reads the finished `patent_disruption.parquet`, and replaces it
atomically with the six extra columns appended.


In [8]:
# Self-contained: this section reads the finished parquet and the metadata and can be run alone
# (python run_notebook.py <nb> --from <this code cell> --save-outputs). It appends the cohort
# percentile columns and replaces the file atomically; re-running recomputes them.
import os, time, duckdb
import numpy as np, pandas as pd
t0 = time.time()
_mb = os.environ.get('SLURM_MEM_PER_NODE')
DUCKDB_MEM = os.environ.get('NB_DUCKDB_MEM') or (f'{int(int(_mb) * 0.8)}MB' if _mb else '16GB')
con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEM}'"); con.execute(f"SET threads={int(os.environ.get('SLURM_CPUS_PER_TASK', '4'))}")
con.execute("SET temp_directory='/project/jevans/Dawoon/Science of Science/.duckdb_tmp'"); con.execute('SET preserve_insertion_order=false')
HORIZONS = ['3', '5', '10', 'all']

OUT_FP = '/project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption.parquet'
META_FP = '/project/jevans/Dawoon/Science of Science/PatentView/output/patent_metadata.parquet'
TMP_FP = OUT_FP + '.pctl.tmp.parquet'
cols = con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') LIMIT 0").df().columns.tolist()
base_cols = [c for c in cols if not (c.endswith('_pctl') or c.endswith('_pctl_cume')) and c not in ('pctl_year', 'pctl_group')]
n0 = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'{n0:,} rows; base columns {len(base_cols)}; existing percentile columns: {len(cols) - len(base_cols)}')
con.execute(f"""
CREATE TEMP TABLE key AS
SELECT CAST(patent_id AS VARCHAR) AS patent_id, filing_year AS pctl_year,
       CASE WHEN upper(substr(trim(cpc_code), 1, 1)) IN ('A','B','C','D','E','F','G','H','Y') THEN upper(substr(trim(cpc_code), 1, 1)) END AS pctl_group
FROM read_parquet('{META_FP}') WHERE filing_year IS NOT NULL""")
# Two percentiles per window: CD_h_pctl = minimum rank / n (share strictly below), and CD_h_pctl_cume = share
# at or below the value (ties inclusive, like cume_dist). CD has large tie blocks at its extremes (CD = 1 when every
# citer cites only the focal document), so the minimum-rank percentile of the whole top block can sit far below 0.9;
# the cumulative percentile is the one to threshold for "top x%" hits.
pctl = ', '.join(
    f"CASE WHEN d.CD_{h} IS NULL OR k.pctl_group IS NULL THEN NULL "
    f"ELSE rank() OVER (PARTITION BY k.pctl_year, k.pctl_group ORDER BY d.CD_{h}) * 1.0 "
    f"/ count(d.CD_{h}) OVER (PARTITION BY k.pctl_year, k.pctl_group) END AS CD_{h}_pctl, "
    f"CASE WHEN d.CD_{h} IS NULL OR k.pctl_group IS NULL THEN NULL "
    f"ELSE count(d.CD_{h}) OVER (PARTITION BY k.pctl_year, k.pctl_group ORDER BY d.CD_{h} RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 1.0 "
    f"/ count(d.CD_{h}) OVER (PARTITION BY k.pctl_year, k.pctl_group) END AS CD_{h}_pctl_cume" for h in HORIZONS)
print(f'[{time.time() - t0:.0f}s] computing percentiles')
con.execute(f"""
COPY (
  SELECT {', '.join('d.' + c for c in base_cols)}, k.pctl_year, k.pctl_group, {pctl}
  FROM (SELECT * REPLACE (CAST(patent_id AS VARCHAR) AS patent_id) FROM read_parquet('{OUT_FP}')) d LEFT JOIN key k USING (patent_id)
) TO '{TMP_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
chk = con.execute(f"""
SELECT count(*) AS rows, count(pctl_group) AS with_group,
       {', '.join(f'count(CD_{h}_pctl) AS n_pctl_{h}, min(CD_{h}_pctl) AS min_{h}, max(CD_{h}_pctl) AS max_{h}, max(CD_{h}_pctl_cume) AS max_cume_{h}, sum(CASE WHEN CD_{h}_pctl_cume < CD_{h}_pctl THEN 1 ELSE 0 END) AS cume_below_min_{h}' for h in HORIZONS)},
       sum(CASE WHEN CD_5 IS NOT NULL AND pctl_group IS NOT NULL AND CD_5_pctl IS NULL THEN 1 ELSE 0 END) AS unexpected_null_5
FROM read_parquet('{TMP_FP}')""").df()
display(chk.T)
assert chk.rows[0] == n0, 'row count changed'
assert chk.unexpected_null_5[0] == 0 and chk.max_5[0] <= 1 and chk.min_5[0] > 0
assert chk.max_cume_5[0] <= 1 + 1e-12 and chk.cume_below_min_5[0] == 0, 'cumulative percentile must be >= minimum-rank percentile'
os.replace(TMP_FP, OUT_FP)
con.close()
print(f'[{time.time() - t0:.0f}s] wrote {OUT_FP} ({os.path.getsize(OUT_FP) / 1e9:.2f} GB): {len(base_cols)} + 10 columns')


8,062,166 rows; base columns 29; existing percentile columns: 6
[1s] computing percentiles


,0
rows,8.062166e+06
with_group,8.045326e+06
n_pctl_3,5.606134e+06
min_3,1.904435e-05
max_3,1.000000e+00
max_cume_3,1.000000e+00
cume_below_min_3,0.000000e+00
n_pctl_5,5.767795e+06
min_5,1.879805e-05
max_5,1.000000e+00


[8s] wrote /project/jevans/Dawoon/Science of Science/PatentView/output/patent_disruption.parquet (0.31 GB): 29 + 10 columns
